# 16: Definitive duplicate check across all 88,702 EyePACS images

Notebook 15c reported 2,243 'exact hash matches' between the new Kaggle test images and the
35,126 already used. That figure is an artefact: it used a 64-bit perceptual hash on a 32x32
grayscale thumbnail, and fundus photographs (circular discs on black backgrounds) collide
heavily at that resolution. The examples give it away, with single new images matching four to
six old ones, which duplication cannot produce.

This notebook replaces it with an exact test. Both sets went through identical preprocessing
(RGB, 512x512 Lanczos), so a genuinely reused image is **pixel-identical**, and a SHA-256 over
decoded pixels decides it with no threshold.

It reports:
1. **Exact duplicates across sets** (new test images versus all previously used), by pixel hash.
2. **Duplicates within each set**, which calibrates the result: the Kaggle training release is
   known to contain some repeated images, so a nonzero within-set count is expected and tells us
   what 'normal' looks like.
3. **Collision diagnostics** for the old 64-bit hash, to confirm the earlier number was spurious.
4. A **256-bit perceptual hash** for near-duplicates (same eye, slightly different capture),
   with every candidate verified by direct pixel comparison.

No model is run and nothing is evaluated. CPU runtime with many cores; about 30 to 45 minutes.
Run cells 1-5.

In [1]:
# Cell 1: mount and locate the image archives
from google.colab import drive
drive.mount('/content/drive')
import os, io, json, zipfile, hashlib, time, datetime
import numpy as np, pandas as pd
from PIL import Image
from scipy.fftpack import dct

BASE='/content/drive/MyDrive/stochastic_orthogonality/data'
V2=f'{BASE}/dr_ensemble_v2'
OLD=[f'{BASE}/eyepacs_full/eyepacs_processed_batch{i}.zip' for i in range(1,6)]          # 19,332 used in v2 splits
OLD+= [f'{V2}/06_fresh_images/{f}' for f in sorted(os.listdir(f'{V2}/06_fresh_images')) if f.endswith('.zip')]
NEW = [f'{V2}/07_kaggle_test_images/{f}' for f in sorted(os.listdir(f'{V2}/07_kaggle_test_images')) if f.endswith('.zip')]
for p in OLD+NEW: print(('OK   ' if os.path.exists(p) else 'MISS '), os.path.basename(p))
print(f'\nold archives {len(OLD)} | new archives {len(NEW)} | cores {os.cpu_count()}')

Mounted at /content/drive
OK    eyepacs_processed_batch1.zip
OK    eyepacs_processed_batch2.zip
OK    eyepacs_processed_batch3.zip
OK    eyepacs_processed_batch4.zip
OK    eyepacs_processed_batch5.zip
OK    eyepacs_fresh_batch1_v2.zip
OK    eyepacs_fresh_batch2_v2.zip
OK    eyepacs_fresh_batch3_v2.zip
OK    eyepacs_fresh_batch4_v2.zip
OK    eyepacs_kaggletest_batch1_v2.zip
OK    eyepacs_kaggletest_batch2_v2.zip
OK    eyepacs_kaggletest_batch3_v2.zip
OK    eyepacs_kaggletest_batch4_v2.zip

old archives 9 | new archives 4 | cores 2


In [2]:
# Cell 2: hash every image: exact pixel SHA-256, a 256-bit pHash, and the old 64-bit pHash
from multiprocessing import Pool

def phash_bits(arr, hs):
    side = hs*4
    a = np.asarray(Image.fromarray(arr).convert('L').resize((side,side), Image.LANCZOS), dtype=np.float64)
    d = dct(dct(a, axis=0, norm='ortho'), axis=1, norm='ortho')[:hs,:hs]
    m = np.median(d.ravel()[1:])
    return ''.join('1' if b else '0' for b in (d>m).ravel())

def hash_one(args):
    path, name = args
    with zipfile.ZipFile(path) as z:
        raw = z.read(name)
    arr = np.asarray(Image.open(io.BytesIO(raw)).convert('RGB'))
    return (os.path.splitext(os.path.basename(name))[0],
            hashlib.sha256(arr.tobytes()).hexdigest(),   # exact pixel identity
            phash_bits(arr, 16),                          # 256-bit perceptual
            phash_bits(arr, 8))                           # the old 64-bit, for diagnostics

def hash_archives(paths, label):
    jobs=[]
    for p in paths:
        with zipfile.ZipFile(p) as z:
            jobs += [(p,n) for n in z.namelist() if n.lower().endswith('.png')]
    print(f'{label}: {len(jobs)} images', flush=True)
    out={}; t0=time.time()
    with Pool(os.cpu_count()) as pool:
        for k,(iid,sh,p256,p64) in enumerate(pool.imap_unordered(hash_one, jobs, chunksize=16),1):
            out[iid]={'sha':sh,'p256':p256,'p64':p64}
            if k%5000==0: print(f'  {label} {k}/{len(jobs)} ({time.time()-t0:.0f}s)', flush=True)
    return out

H_OLD = hash_archives(OLD, 'previously used')
H_NEW = hash_archives(NEW, 'new kaggle test')
print(f'\nhashed: old {len(H_OLD)}, new {len(H_NEW)}, total {len(H_OLD)+len(H_NEW)}')

previously used: 35126 images
  previously used 5000/35126 (759s)
  previously used 10000/35126 (1388s)
  previously used 15000/35126 (2049s)
  previously used 20000/35126 (2674s)
  previously used 25000/35126 (3279s)
  previously used 30000/35126 (3885s)
  previously used 35000/35126 (4449s)
new kaggle test: 53576 images
  new kaggle test 5000/53576 (599s)
  new kaggle test 10000/53576 (1194s)
  new kaggle test 15000/53576 (1777s)
  new kaggle test 20000/53576 (2365s)
  new kaggle test 25000/53576 (2936s)
  new kaggle test 30000/53576 (3495s)
  new kaggle test 35000/53576 (4082s)
  new kaggle test 40000/53576 (4695s)
  new kaggle test 45000/53576 (5271s)
  new kaggle test 50000/53576 (5877s)

hashed: old 35126, new 53576, total 88702


In [3]:
# Cell 3: exact duplicates, across sets and within each set
def by(d, field):
    m={}
    for k,v in d.items(): m.setdefault(v[field],[]).append(k)
    return m
old_sha=by(H_OLD,'sha'); new_sha=by(H_NEW,'sha')
cross=[(k,old_sha[v['sha']]) for k,v in H_NEW.items() if v['sha'] in old_sha]
within_old=[(h,ks) for h,ks in old_sha.items() if len(ks)>1]
within_new=[(h,ks) for h,ks in new_sha.items() if len(ks)>1]
EXACT={'cross_set_exact_duplicate_images':len(cross),
       'cross_examples':[{'new':k,'old':v} for k,v in cross[:10]],
       'within_previously_used_duplicate_groups':len(within_old),
       'within_previously_used_images_involved':int(sum(len(k) for _,k in within_old)),
       'within_new_duplicate_groups':len(within_new),
       'within_new_images_involved':int(sum(len(k) for _,k in within_new))}
print(json.dumps(EXACT, indent=2))
print('\nWithin-set duplicates calibrate the cross-set number: the Kaggle release is known to')
print('contain some repeated images, so a nonzero within-set count is expected.')

{
  "cross_set_exact_duplicate_images": 0,
  "cross_examples": [],
  "within_previously_used_duplicate_groups": 0,
  "within_previously_used_images_involved": 0,
  "within_new_duplicate_groups": 0,
  "within_new_images_involved": 0
}

Within-set duplicates calibrate the cross-set number: the Kaggle release is known to
contain some repeated images, so a nonzero within-set count is expected.


In [4]:
# Cell 4: was the earlier 64-bit number a collision artefact? and near-duplicate search
u64_old=len(set(v['p64'] for v in H_OLD.values())); u256_old=len(set(v['p256'] for v in H_OLD.values()))
u64_new=len(set(v['p64'] for v in H_NEW.values())); u256_new=len(set(v['p256'] for v in H_NEW.values()))
old64=by(H_OLD,'p64'); old256=by(H_OLD,'p256')
cross64=[k for k,v in H_NEW.items() if v['p64'] in old64]
cross256=[(k,old256[v['p256']]) for k,v in H_NEW.items() if v['p256'] in old256]
DIAG={'unique_64bit_hashes_old':u64_old,'images_old':len(H_OLD),
      'unique_64bit_hashes_new':u64_new,'images_new':len(H_NEW),
      'unique_256bit_hashes_old':u256_old,'unique_256bit_hashes_new':u256_new,
      'cross_matches_64bit':len(cross64),'cross_matches_256bit':len(cross256)}
print(json.dumps(DIAG, indent=2))
print('\nIf the 64-bit hash has far fewer unique values than images, it collides and the earlier')
print('2,243 figure was spurious. The 256-bit count is the meaningful near-duplicate signal.')

# verify every 256-bit candidate by direct pixel comparison
def load(iid, archives):
    for p in archives:
        with zipfile.ZipFile(p) as z:
            n=f'{iid}.png'
            if n in z.namelist():
                return np.asarray(Image.open(io.BytesIO(z.read(n))).convert('RGB'), dtype=np.int16)
    return None
VER=[]
for k, olds in cross256[:200]:
    a=load(k,NEW)
    for o in olds[:3]:
        b=load(o,OLD)
        if a is None or b is None or a.shape!=b.shape: continue
        d=np.abs(a-b)
        VER.append({'new':k,'old':o,'mean_abs_diff':round(float(d.mean()),3),'max_abs_diff':int(d.max())})
V=pd.DataFrame(VER)
if len(V):
    print('\npixel verification of 256-bit candidates (up to 200):')
    print(V.sort_values('mean_abs_diff').head(15).to_string(index=False))
    print(f"\ncandidates with mean abs diff < 1 (true duplicates): {int((V['mean_abs_diff']<1).sum())} of {len(V)}")
else:
    print('\nno 256-bit cross-set candidates to verify')

{
  "unique_64bit_hashes_old": 34183,
  "images_old": 35126,
  "unique_64bit_hashes_new": 51806,
  "images_new": 53576,
  "unique_256bit_hashes_old": 35126,
  "unique_256bit_hashes_new": 53576,
  "cross_matches_64bit": 2243,
  "cross_matches_256bit": 0
}

If the 64-bit hash has far fewer unique values than images, it collides and the earlier
2,243 figure was spurious. The 256-bit count is the meaningful near-duplicate signal.

no 256-bit cross-set candidates to verify


In [5]:
# Cell 5: verdict and record; supersede the 15c overlap figure
verdict=('no reused images between the new Kaggle test set and anything already used'
         if EXACT['cross_set_exact_duplicate_images']==0 else
         f"{EXACT['cross_set_exact_duplicate_images']} reused images found; exclude them before use")
REC={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'supersedes':'overlap_check in kaggletest_preparation_v2.json (64-bit perceptual hash, collision-prone)',
     'method':'SHA-256 over decoded 512x512 RGB pixels; both sets preprocessed identically, so a '
              'reused image is pixel-identical. 256-bit perceptual hash for near-duplicates, each '
              'candidate verified by direct pixel comparison.',
     'exact':EXACT,'hash_diagnostics':DIAG,
     'pixel_verified_candidates':VER[:50],'verdict':verdict,
     'limitation':'pixel identity detects reused images; it cannot detect the same patient '
                  'photographed on a different date. Kaggle documents train and test as different patients.',
     'note':'No model run, no evaluation.'}
with open(f'{V2}/99_audit/duplicate_check_definitive_v2.json','w') as f: json.dump(REC,f,indent=2,default=str)
if len(V): V.to_csv(f'{V2}/99_audit/duplicate_candidates_verified_v2.csv', index=False)
print('VERDICT:', verdict)
print('\nwrote 99_audit/duplicate_check_definitive_v2.json')

VERDICT: no reused images between the new Kaggle test set and anything already used

wrote 99_audit/duplicate_check_definitive_v2.json
